# AnswerCarefully — word splits and yes/はい flakes (no LLM)

Reads `My Drive/ez-penta/answercarefully.sqlite3` and writes a new `My Drive/ez-penta/answercarefully_simple.sqlite3` with these columns added right after `text_en`. It needs no model or GPU, so any Colab server works (**Select Kernel → Colab → New Colab Server**, CPU is fine) and it finishes in seconds.

| Column | What it holds |
|---|---|
| `text_words` | Japanese prompt split into words by MeCab (fugashi + UniDic), as a JSON list |
| `text_en_words` | English prompt split with `str.split()`, as a JSON list |
| `flake_every10_{ja,en}_{yes,hai}` | `yes` or `はい` inserted after every 10th character of the Japanese (`ja`) or English (`en`) prompt |
| `flake_quartile_{ja,en}_{yes,hai}` | exactly 4 flakes, at 25%, 50%, 75% and 100% of the prompt's length |

Flakes are inserted at exact character positions with no spaces added, so they can land inside a word (`talyesk`). Positions are counted in the original prompt, so the flakes don't shift each other.

In [ ]:
# unidic-lite ships no wheel, so it stays out of requirements.txt (run.sh installs binaries only).
%pip install -q fugashi==1.5.2 unidic-lite==1.0.8

In [ ]:
import json
import shutil
import sqlite3
from pathlib import Path

import pandas as pd

DRIVE_DIR = Path("/content/drive/MyDrive/ez-penta")
DB_IN = DRIVE_DIR / "answercarefully.sqlite3"
DB_OUT = DRIVE_DIR / "answercarefully_simple.sqlite3"
WORK_DIR = Path("/content/work")
INTERVAL = 10  # characters between flakes in the flake_every10_* columns
FLAKES = {"yes": "yes", "hai": "はい"}

if not DRIVE_DIR.exists():
    from google.colab import drive
    drive.mount("/content/drive")
if not DB_IN.exists():
    raise FileNotFoundError(f"Upload answercarefully.sqlite3 to {DRIVE_DIR} in Google Drive, then rerun this cell.")

# SQLite is unreliable directly on the Drive mount, so read and write local copies.
WORK_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_IN = WORK_DIR / DB_IN.name
LOCAL_OUT = WORK_DIR / DB_OUT.name
shutil.copy(DB_IN, LOCAL_IN)
print(f"Copied {DB_IN} -> {LOCAL_IN}")

connection = sqlite3.connect(LOCAL_IN)
records = pd.read_sql_query("SELECT * FROM records", connection)
connection.close()
print(f"{len(records)} records")

## 1. Word splits

Japanese has no spaces, so it needs a morphological analyzer. MeCab with the UniDic dictionary (`fugashi[unidic-lite]`) is the standard choice. It splits into short units, so particles and endings are separate words: `DMでお話ししたい` → `DM / で / お / 話し / し / たい`. MeCab skips whitespace, so spaces and line breaks in the prompt don't appear in the list.

In [ ]:
import fugashi

tagger = fugashi.Tagger()


def japanese_words(text):
    # MeCab keeps full-width spaces (U+3000) as tokens; drop them like ordinary whitespace.
    return [word.surface for word in tagger(text) if not word.surface.isspace()]


records["text_words"] = records["text"].map(japanese_words)
records["text_en_words"] = records["text_en"].map(lambda text: text.split() if isinstance(text, str) else None)
records[["text", "text_words", "text_en", "text_en_words"]].head()

## 2. Flakes

In [ ]:
def insert_flakes(text, positions, flake):
    """Insert flake after each position, counted in the original text; positions must be ascending."""
    pieces, previous = [], 0
    for position in positions:
        pieces += [text[previous:position], flake]
        previous = position
    pieces.append(text[previous:])
    return "".join(pieces)


def every_n(text, n=INTERVAL):
    # After the 10th, 20th, ... character, including the end when the length is a multiple of n.
    return range(n, len(text) + 1, n)


def quartiles(text):
    # Always 4 positions; a very short prompt can get two flakes at the same spot.
    return [len(text) * k // 4 for k in range(1, 5)]


FLAKE_COLUMNS = []
for pattern, positions in (("every10", every_n), ("quartile", quartiles)):
    for base, source in (("ja", "text"), ("en", "text_en")):
        for flake_name, flake in FLAKES.items():
            column = f"flake_{pattern}_{base}_{flake_name}"
            records[column] = records[source].map(
                lambda text: insert_flakes(text, positions(text), flake) if isinstance(text, str) else None
            )
            FLAKE_COLUMNS.append(column)
FLAKE_COLUMNS

In [ ]:
# Check: each column has exactly the expected number of flakes added.
for column in FLAKE_COLUMNS:
    _, pattern, base, flake_name = column.split("_")
    source = records["text" if base == "ja" else "text_en"]
    expected = source.str.len() + source.map(lambda text: len(every_n(text)) if pattern == "every10" else 4) * len(FLAKES[flake_name])
    assert (records[column].str.len() == expected).all(), column
print("All flake columns have the expected length.")

In [ ]:
pd.set_option("display.max_colwidth", None)
records[["text", "text_en", *FLAKE_COLUMNS]].head(3).T

## 3. Save as a new SQLite db (and copy it to Drive)

`answercarefully_simple.sqlite3` keeps every original column and the same primary key; the new columns go right after `text_en`.

In [ ]:
from datetime import datetime, timezone
from importlib.metadata import version

NEW_COLUMNS = ["text_words", "text_en_words", *FLAKE_COLUMNS]
saved = records.copy()
for column in ["text_words", "text_en_words"]:
    saved[column] = saved[column].map(lambda words: json.dumps(words, ensure_ascii=False) if words is not None else None)

shutil.copy(LOCAL_IN, LOCAL_OUT)
connection = sqlite3.connect(LOCAL_OUT)
table_info = connection.execute("PRAGMA table_info(records)").fetchall()

definitions, order = [], []
for _, name, column_type, not_null, _, _ in table_info:
    definitions.append(f'"{name}" {column_type}' + (" NOT NULL" if not_null else ""))
    order.append(name)
    if name == "text_en":
        definitions += [f'"{column}" TEXT' for column in NEW_COLUMNS]
        order += NEW_COLUMNS
primary_key = [info[1] for info in sorted(table_info, key=lambda info: info[5]) if info[5]]

rows = ([None if pd.isna(value) else value for value in row] for row in saved[order].itertuples(index=False))
with connection:
    connection.execute("DROP TABLE records")
    connection.execute(f"CREATE TABLE records ({', '.join(definitions)}, PRIMARY KEY ({', '.join(primary_key)}))")
    connection.executemany(f"INSERT INTO records ({', '.join(order)}) VALUES ({', '.join('?' * len(order))})", rows)
    connection.execute("DROP TABLE IF EXISTS generation_info")
    connection.execute("CREATE TABLE generation_info (columns TEXT NOT NULL, method TEXT NOT NULL, created_at TEXT NOT NULL)")
    connection.execute("INSERT INTO generation_info VALUES (?, ?, ?)", (
        ", ".join(NEW_COLUMNS),
        f"fugashi {version('fugashi')} + unidic-lite {version('unidic-lite')} word split; str.split(); flakes every {INTERVAL} chars and at quartiles",
        datetime.now(timezone.utc).isoformat(),
    ))
print([name for _, name, *_ in connection.execute("PRAGMA table_info(records)")])
connection.close()

shutil.copy(LOCAL_OUT, DB_OUT)
print(f"Saved {DB_OUT}")